# Reproduction notebook — distant-stage colorectal neuroendocrine carcinoma (SEER landmark analysis)

Companion notebook of the repository **`colorectal-nec-landmark-seer`** for

> Wang J, Zhang S, Liang W, Choi W. *Conditional one-year cancer mortality shows no early decline in distant-stage colorectal neuroendocrine carcinoma: a SEER landmark analysis with cross-tumor comparison.* Submitted to *International Journal of Cancer* (Research Article, Cancer Epidemiology), 2026.
>
> Corresponding author: **Wansuk Choi**, Department of Physical Therapy, College of Health Sciences, Kyungwoon University, Gumi, Republic of Korea · wschoi@ikw.ac.kr · ORCID 0000-0003-1324-6723.

## 0. Purpose and data-governance notice (read before running)

**What it does.** Starting from *your own* four SEER\*Stat case-listing exports, the notebook installs the Python and R dependencies, checks the exports (counts only), runs the repository pipeline step by step — cohort construction, analyses A01–A12, manuscript Figures 1–3 and Supplementary Figures S1–S7 — and verifies the output against the manuscript (`tests/check_anchors.py`) and against the authors' aggregate reference results (`tests/compare_with_reference.py`). It runs in **Google Colab** or in a **local Jupyter** session (also headless with `jupyter nbconvert --execute`).

| Mode | Data | Bootstrap | Purpose |
|---|---|---|---|
| `full` | your SEER exports | B = 2,000 (paper settings) | exact reproduction; anchors and reference comparison must pass |
| `quick` | your SEER exports | B = 40 (smoke test) | fast check of the set-up; point estimates/anchors pass, bootstrap CIs differ from the paper |
| `synthetic` | random files generated by `tools/make_synthetic_seer_export.py` | B = 40 | runs end to end **without SEER access**; results are meaningless and mismatches are expected |

> ### ⚠️ SEER Research Data Use Agreement (DUA)
> * The repository and this notebook contain **no SEER data**. The SEER DUA forbids sharing or redistributing SEER data, including derived patient-level files (`work/cohort_*.csv`).
> * **Running in Colab processes SEER data on Google cloud infrastructure** (Colab virtual machine; Google Drive if you store the exports there). Whether this is permitted is for **each user to judge against their own DUA and institutional rules**. If in doubt, **use local Jupyter** (recommended) or the `synthetic` mode. In Colab, `full`/`quick` therefore require you to tick `I_CONFIRM_DUA_PERMITS_CLOUD`.
> * The notebook **never prints or saves record-level rows**: the data check shows only file sizes, row counts and column counts, and step logs show only aggregate messages. Do not add cells that display data rows (`head()`, `print(df)`, etc.).
> * The optional download (Section 6) packs **aggregate** outputs only (JSON/Markdown/aggregate CSV/figures). Outputs in `results/` are unsuppressed working files and include a few cells based on 1–4 patients (e.g. histology codes with one case); before publishing or sharing any of them, run `python3 tools/suppress_small_cells.py` (SEER DUA item 14: statistics on demographic characteristics or geography based on 1–4 patients must be suppressed). `reference_results/` already holds publication-safe copies.
> * **Delete runtime files after use**: the last cell removes `work/` (derived cohorts) and any data copied or generated in the runtime. In Colab, also end the session (*Runtime ▸ Disconnect and delete runtime*).

**Obtaining the data.** Request SEER access at <https://seer.cancer.gov/data/access.html>, accept the DUA, and create the four case-listing exports in SEER\*Stat exactly as specified in [`data_spec/SEER_export_specification.md`](../data_spec/SEER_export_specification.md): database *Incidence – SEER Research Data, 17 Registries, Nov 2025 Sub (2000–2023)*, 101 variables, quoted labels, tab-delimited, CRLF, header line:

| File (default name) | Content |
|---|---|
| `colorectal_NEN_SEER17_2000_2023_full_20260806.txt` | colorectal neuroendocrine neoplasms (source of the NEC cohorts) |
| `colorectal_adenocarcinoma_SEER17_2000_2023_full_20260928.txt` | colorectal adenocarcinoma (comparator) |
| `SCLC_lung_SEER17_2000_2023_full_20260928.txt` | lung small-cell / large-cell neuroendocrine carcinoma (comparators) |
| `pancreatic_adenocarcinoma_SEER17_2000_2023_full_20260928.txt` | pancreatic adenocarcinoma (comparator) |

Other file names can be used via `NEC_FILE_NEN`, `NEC_FILE_AC`, `NEC_FILE_LUNG`, `NEC_FILE_PANC`. See also `docs/output_map.md` (script → table/figure) and `docs/data_availability.md`.

**How to run.** *Colab:* set the form in Section 1, then *Runtime ▸ Run all*. *Local:* `NEC_REPO_DIR=/path/to/colorectal-nec-landmark-seer NEC_DATA_DIR=/path/to/exports NEC_MODE=full jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 notebooks/reproduce_colab.ipynb` (or open it in Jupyter). Expected time: ≈ 12–17 min for `full` on a 10-core Apple-silicon laptop; on a free Colab runtime (2 vCPUs) expect substantially longer (see `notebooks/README.md`).

---

### 요약 (Korean summary)

* 이 노트북은 **사용자 본인의 SEER\*Stat 내보내기 파일 4개**로 논문의 모든 결과(코호트 구축 → A01–A12 분석 → 그림 1–3, 보충 그림 S1–S7)를 재현하고, 원고 수치(`check_anchors.py`) 및 저자의 집계 참조 결과(`compare_with_reference.py`)와 대조합니다.
* 모드: `full`(논문 설정, B = 2,000), `quick`(B = 40, 빠른 점검), `synthetic`(SEER 접근 없이 무작위 합성 데이터로 실행 — 결과가 논문과 다른 것이 정상).
* **SEER 데이터 이용 약정(DUA) 주의:** Colab에서 실행하면 SEER 데이터가 Google 클라우드(Colab VM, Google Drive)에서 처리됩니다. 허용 여부는 각 사용자가 자신의 DUA와 기관 규정에 따라 판단해야 하며, 확실하지 않으면 **로컬 Jupyter** 실행을 권장합니다. 노트북은 개별 레코드(행)를 출력·저장하지 않으며(행 수·열 수만 표시), 사용 후 마지막 셀이 `work/`(파생 코호트)와 런타임에 복사·생성된 데이터를 삭제합니다.
* 데이터 획득 및 정확한 내보내기 사양: `data_spec/SEER_export_specification.md`.
* 교신저자: 최완석(Wansuk Choi), 경운대학교 물리치료학과 (Department of Physical Therapy, College of Health Sciences, Kyungwoon University), wschoi@ikw.ac.kr.

## 1. Environment detection and parameters

In **Colab**, fill in the form below (the defaults run the safe `synthetic` mode). Locally, the same parameters are read from environment variables, so the notebook can be executed headless:

| Parameter | Colab form | Local environment variable (default) |
|---|---|---|
| repository | `REPO_SOURCE` = `github` (`REPO_URL`, optional `REPO_REF`) or `drive` (`REPO_DRIVE_PATH`) | `NEC_REPO_DIR` (parent of this notebook's folder) |
| SEER exports | `DATA_DIR` | `NEC_DATA_DIR` (`<repo>/data/raw`) |
| mode | `MODE` = `full` / `quick` / `synthetic` | `NEC_MODE` (`full`) |
| bootstrap override | `BOOT` (empty = mode default) | `NEC_BOOT` (empty) |
| zip aggregate outputs | `MAKE_ZIP` | `NEC_MAKE_ZIP` (`0`) |
| delete derived cohorts at the end | `CLEANUP_AT_END` | `NEC_CLEANUP` (`1`) |

In [ ]:
import os, re, sys, time, json, shutil, subprocess, platform, importlib.util, zipfile, glob
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f"Python {platform.python_version()} on {platform.platform()}  |  running in Colab: {IN_COLAB}  |  CPUs: {os.cpu_count()}")

In [ ]:
#@title Parameters (Colab form; ignored when run locally, see the table above) { display-mode: "form" }
MOUNT_DRIVE = True  #@param {type:"boolean"}
REPO_SOURCE = "github"  #@param ["github", "drive"]
REPO_URL = "https://github.com/<account>/colorectal-nec-landmark-seer.git"  #@param {type:"string"}
REPO_REF = ""  #@param {type:"string"}
REPO_DRIVE_PATH = "/content/drive/MyDrive/.../02_GitHub_Repository/colorectal-nec-landmark-seer"  #@param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/SEER_exports"  #@param {type:"string"}
COPY_DATA_TO_RUNTIME = True  #@param {type:"boolean"}
MODE = "synthetic"  #@param ["full", "quick", "synthetic"]
BOOT = ""  #@param {type:"string"}
I_CONFIRM_DUA_PERMITS_CLOUD = False  #@param {type:"boolean"}
PIN_PY_VERSIONS = True  #@param {type:"boolean"}
MAKE_ZIP = False  #@param {type:"boolean"}
CLEANUP_AT_END = True  #@param {type:"boolean"}
SYNTH_SCALE = 1.0  #@param {type:"number"}
HEARTBEAT_SECONDS = 60  #@param {type:"integer"}

In [ ]:
def _truthy(v):
    return str(v).strip().lower() in ("1", "true", "yes", "y", "on")

if not IN_COLAB:
    # ---- local Jupyter / nbconvert: parameters from environment variables ----
    _here = Path.cwd()
    _guess = _here.parent if (_here.parent / "run_all.py").exists() else _here
    REPO_SOURCE = "local"
    REPO_DIR = Path(os.environ.get("NEC_REPO_DIR", "") or _guess).expanduser().resolve()
    DATA_DIR = os.environ.get("NEC_DATA_DIR", "") or str(REPO_DIR / "data" / "raw")
    MODE = (os.environ.get("NEC_MODE", "") or "full").strip().lower()
    BOOT = os.environ.get("NEC_BOOT", "").strip()
    MAKE_ZIP = _truthy(os.environ.get("NEC_MAKE_ZIP", "0"))
    CLEANUP_AT_END = _truthy(os.environ.get("NEC_CLEANUP", "1"))
    SYNTH_SCALE = float(os.environ.get("NEC_SYNTH_SCALE", "1.0") or 1.0)
    HEARTBEAT_SECONDS = int(os.environ.get("NEC_HEARTBEAT", "60") or 60)
    COPY_DATA_TO_RUNTIME = False
    RUNTIME_DIR = REPO_DIR / "data" / "notebook_runtime"      # inside data/ -> git-ignored
else:
    # ---- Google Colab ----
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
    RUNTIME_DIR = Path("/content/nec_runtime")
    REPO_DIR = Path("/content/colorectal-nec-landmark-seer")   # always run from the VM's local disk, never inside Drive
    if REPO_DIR.exists():
        print(f"Using the repository already present at {REPO_DIR} (delete it to fetch a fresh copy).")
    elif REPO_SOURCE == "github":
        if "<account>" in REPO_URL:
            raise ValueError("Set REPO_URL to the GitHub address of the repository (replace <account>), or use REPO_SOURCE='drive'.")
        subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
        if REPO_REF.strip():
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF.strip()], check=True)
    elif REPO_SOURCE == "drive":
        src = Path(REPO_DRIVE_PATH)
        if not (src / "run_all.py").exists():
            raise FileNotFoundError(f"run_all.py not found in REPO_DRIVE_PATH={src}; point it to the repository folder copied to Drive.")
        # copy code only: never carry data/, work/ or results/ from Drive into the runtime
        shutil.copytree(src, REPO_DIR, ignore=shutil.ignore_patterns("work", "results", "data", ".git", ".venv", "__pycache__", "*.zip"))
        (REPO_DIR / "data" / "raw").mkdir(parents=True, exist_ok=True)
    else:
        raise ValueError("REPO_SOURCE must be 'github' or 'drive'")

MODE = str(MODE).strip().lower()
if MODE not in ("full", "quick", "synthetic"):
    raise ValueError(f"MODE must be 'full', 'quick' or 'synthetic' (got {MODE!r})")
if IN_COLAB and MODE in ("full", "quick") and not I_CONFIRM_DUA_PERMITS_CLOUD:
    raise PermissionError("MODE='full'/'quick' processes SEER data on Google cloud. Tick I_CONFIRM_DUA_PERMITS_CLOUD only if your "
                          "SEER Data Use Agreement and institution allow this; otherwise run locally or use MODE='synthetic'.")
if not (REPO_DIR / "run_all.py").exists():
    raise FileNotFoundError(f"{REPO_DIR} is not the colorectal-nec-landmark-seer repository (run_all.py missing); set NEC_REPO_DIR.")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

# ---- pipeline configuration (the same environment variables as run_all.py / python/config.py / R/config.R) ----
WORK_DIR = REPO_DIR / "work"                       # patient-level derived cohorts: deleted by the last cell
RESULTS_DIR = REPO_DIR / "results"                 # aggregate results
FIG_DIR = REPO_DIR / "figures" / "output"          # figures; manuscript figures in manuscript_final/
if MODE == "synthetic":
    DATA_DIR_EFFECTIVE = RUNTIME_DIR / "synthetic_seer_exports"
else:
    DATA_DIR_EFFECTIVE = Path(DATA_DIR).expanduser().resolve()   # may be replaced by a runtime copy in Section 3 (Colab)

PIPE_ENV = {
    "NEC_DATA_DIR": str(DATA_DIR_EFFECTIVE), "NEC_WORK_DIR": str(WORK_DIR),
    "NEC_RESULTS_DIR": str(RESULTS_DIR), "NEC_FIG_DIR": str(FIG_DIR),
    "NEC_QUICK": "0" if MODE == "full" else "1",
    "NEC_CORES": str(max(1, min(4, os.cpu_count() or 1))),
    "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "PYTHONUNBUFFERED": "1",
}
if str(BOOT).strip():
    PIPE_ENV["NEC_BOOT"] = str(int(str(BOOT).strip()))
else:
    os.environ.pop("NEC_BOOT", None)
os.environ.update(PIPE_ENV)          # the kernel (Section 5 imports) sees exactly what the pipeline sees

print(f"repository     {REPO_DIR}  (source: {REPO_SOURCE})")
print(f"mode           {MODE}   bootstrap: {PIPE_ENV.get('NEC_BOOT', '2000 (paper)' if MODE == 'full' else '40 (quick default)')}")
print(f"SEER exports   {DATA_DIR_EFFECTIVE}" + ("   (synthetic files will be generated here)" if MODE == "synthetic" else ""))
print(f"work (cohorts) {WORK_DIR}   <- patient-level, deleted at the end" )
print(f"results        {RESULTS_DIR}\nfigures        {FIG_DIR}\nruntime files  {RUNTIME_DIR}")

## 2. Dependencies

Python: `requirements.txt` (numpy 2.2.0, pandas 2.3.3, scipy 1.17.1, matplotlib 3.10.8 in the verified run). In Colab the pinned versions are installed with pip (quiet); locally nothing is installed if the packages are importable (use a virtual environment, see the repository README).
R: `Rscript` (R ≥ 4.3; verified with 4.5.3) with data.table, survival, sandwich, jsonlite, cmprsk and muhaz, installed by the repository's `R/install_packages.R` from the CRAN cloud mirror with parallel compilation (`Ncpus`). If R is missing in Colab it is installed with `apt-get`. First-time compilation of the R packages in Colab takes a few minutes.

In [ ]:
REQ = REPO_DIR / "requirements.txt"
PY_PKGS = ("numpy", "pandas", "scipy", "matplotlib")
_preloaded = {p: sys.modules[p].__version__ for p in PY_PKGS if p in sys.modules and hasattr(sys.modules[p], "__version__")}

def _installed_versions():
    code_ = "import json,importlib\nd={}\nfor p in %r:\n    try: d[p]=importlib.import_module(p).__version__\n    except Exception as e: d[p]=None\nprint(json.dumps(d))" % (PY_PKGS,)
    out = subprocess.run([sys.executable, "-c", code_], capture_output=True, text=True)
    return json.loads(out.stdout.strip().splitlines()[-1]) if out.returncode == 0 else {p: None for p in PY_PKGS}

pins = {l.split("==")[0].strip(): l.split("==")[1].strip() for l in REQ.read_text().splitlines() if "==" in l and not l.lstrip().startswith("#")}
t0 = time.time()
if IN_COLAB:
    if PIN_PY_VERSIONS:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ)], capture_output=True, text=True)
        if r.returncode != 0:
            print("Pinned install failed; installing unpinned versions instead (results may differ in the last digits).")
            print("\n".join(r.stderr.strip().splitlines()[-5:]))
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins.keys()], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins.keys()], check=True)
else:
    have = _installed_versions()
    missing = [p for p, v in have.items() if v is None]
    if missing:
        print(f"Missing Python packages {missing}: trying pip install -r requirements.txt into {sys.executable}")
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ)], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("pip install failed (externally managed Python?). Create a virtual environment: "
                               "python3 -m venv .venv && . .venv/bin/activate && pip install -r requirements.txt jupyter, "
                               "then start Jupyter from it.\n" + r.stderr[-1500:])

have = _installed_versions()
for p in PY_PKGS:
    v, pin = have.get(p), pins.get(p)
    print(f"  {p:11s} {str(v):10s} (verified run: {pin})" + ("" if v == pin else "   <- differs from the verified version"))
changed = [p for p, v in _preloaded.items() if have.get(p) != v]
if changed:
    print(f"NOTE: {changed} were already imported by this kernel in another version. The pipeline runs in fresh subprocesses and is "
          "unaffected; if Sections 5-6 raise import errors, use Runtime > Restart session and run all cells again.")
print(f"Python dependencies ready ({time.time() - t0:.0f} s)")

In [ ]:
t0 = time.time()
if shutil.which("Rscript") is None:
    if IN_COLAB:
        print("Rscript not found: installing R with apt-get (about 1-3 min) ...")
        subprocess.run("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y r-base r-base-dev > /dev/null",
                       shell=True, check=True)
    else:
        raise RuntimeError("Rscript is not on the PATH. Install R >= 4.3 (https://cran.r-project.org) and restart Jupyter.")
NCPUS = max(1, os.cpu_count() or 1)
r_env = dict(os.environ, MAKEFLAGS=f"-j{NCPUS}")
print(f"Installing/checking R packages with R/install_packages.R (CRAN cloud mirror, Ncpus = {NCPUS}) ...")
r = subprocess.run(["Rscript", "-e", f"options(Ncpus = {NCPUS}L, repos = c(CRAN = 'https://cloud.r-project.org')); "
                    "source('R/install_packages.R')"], cwd=str(REPO_DIR), env=r_env, capture_output=True, text=True)
lines = (r.stdout + r.stderr).splitlines()
summary = [l for l in lines if "installed" in l and "verified run" in l or l.startswith("R version")]
print("\n".join("  " + l for l in summary) if summary else "\n".join(lines[-20:]))
if r.returncode != 0:
    print("\n".join(lines[-25:]))
    raise RuntimeError("R package installation failed (see the messages above; a missing system library is the usual cause).")
print(f"R dependencies ready ({time.time() - t0:.0f} s)")

## 3. Data check

The four exports must be present in `DATA_DIR`. This cell prints **only** file sizes, numbers of data rows and numbers of columns (and whether the header has the 101 variables of the specification) — never a data row. In `synthetic` mode the files are first generated with `tools/make_synthetic_seer_export.py` (random numbers in the exact export layout, marked as synthetic). In Colab with `COPY_DATA_TO_RUNTIME`, the exports are copied from Drive to the VM's local disk for faster reading (the copy is deleted at the end).

In [ ]:
sys.path.insert(0, str(REPO_DIR / "tools"))
_spec = importlib.util.spec_from_file_location("make_synth", REPO_DIR / "tools" / "make_synthetic_seer_export.py")
make_synth = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(make_synth)
SPEC_COLUMNS = list(make_synth.COLUMNS)            # the 101 variables of data_spec/SEER_export_specification.md
RAW_FILES = {k: os.environ.get(f"NEC_FILE_{k}", v) for k, v in make_synth.FILES.items()}
COPIED_DATA_DIR = None

if MODE == "synthetic":
    if DATA_DIR_EFFECTIVE.exists():
        shutil.rmtree(DATA_DIR_EFFECTIVE)
    t0 = time.time()
    r = subprocess.run([sys.executable, str(REPO_DIR / "tools" / "make_synthetic_seer_export.py"), "--out", str(DATA_DIR_EFFECTIVE),
                        "--scale", str(SYNTH_SCALE)], cwd=str(REPO_DIR), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-2000:])
    print(f"Synthetic exports generated in {time.time() - t0:.0f} s (random data; results will NOT match the paper).")
elif IN_COLAB and COPY_DATA_TO_RUNTIME:
    COPIED_DATA_DIR = RUNTIME_DIR / "seer_exports_copy"
    COPIED_DATA_DIR.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for f in RAW_FILES.values():
        if (DATA_DIR_EFFECTIVE / f).exists():
            shutil.copy2(DATA_DIR_EFFECTIVE / f, COPIED_DATA_DIR / f)
    print(f"Copied the exports to the runtime disk in {time.time() - t0:.0f} s: {COPIED_DATA_DIR} (deleted by the last cell)")
    DATA_DIR_EFFECTIVE = COPIED_DATA_DIR
    PIPE_ENV["NEC_DATA_DIR"] = os.environ["NEC_DATA_DIR"] = str(DATA_DIR_EFFECTIVE)

def count_rows_cols(path, chunk=1 << 24):
    # counts lines in binary chunks; reads only the header line as text (never a data row)
    with open(path, "rb") as fh:
        header = fh.readline()
        n, last = 0, b"\n"
        while True:
            b = fh.read(chunk)
            if not b:
                break
            n += b.count(b"\n"); last = b[-1:]
        if last != b"\n":
            n += 1
    cols = [c.strip().strip('"') for c in header.decode("utf-8-sig", errors="replace").rstrip("\r\n").split("\t")]
    return n, len(cols), cols

missing = [f for f in RAW_FILES.values() if not (DATA_DIR_EFFECTIVE / f).exists()]
if missing:
    raise FileNotFoundError(f"SEER exports not found in {DATA_DIR_EFFECTIVE}:\n  " + "\n  ".join(missing) +
                            "\nSee data_spec/SEER_export_specification.md (or set NEC_FILE_NEN/AC/LUNG/PANC).")
rows = []
for key, f in RAW_FILES.items():
    p = DATA_DIR_EFFECTIVE / f
    n, k, cols = count_rows_cols(p)
    rows.append((key, f, f"{p.stat().st_size / 1e6:,.1f}", f"{n:,}", k, "yes" if cols == SPEC_COLUMNS else ("same set, other order" if set(cols) == set(SPEC_COLUMNS) else "NO")))
w = max(len(r[1]) for r in rows)
print(f"{'export':5s} {'file':{w}s} {'MB':>9s} {'data rows':>11s} {'columns':>8s}  header = specification")
for r_ in rows:
    print(f"{r_[0]:5s} {r_[1]:{w}s} {r_[2]:>9s} {r_[3]:>11s} {r_[4]:>8d}  {r_[5]}")
print("synthetic marker present:", (DATA_DIR_EFFECTIVE / "SYNTHETIC_DATA_README.txt").exists())

## 4. Pipeline, step by step

The steps are those of `run_all.py` (`python3 run_all.py --list`), each run in a fresh subprocess from the repository root with the configuration of Section 1. Full console output goes to `results/logs/NN_STEP.log`; the notebook shows one status line per step with its run time, the step's own reproduction-check lines, and a heartbeat for long steps. In `full` mode the notebook stops at the first failing step (as `run_all.py`); in `quick`/`synthetic` mode it keeps going.

In [ ]:
_spec = importlib.util.spec_from_file_location("nec_run_all", REPO_DIR / "run_all.py")
nec_run_all = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(nec_run_all)
STEPS = {s[0]: s for s in nec_run_all.STEPS}
STEP_IDS = list(STEPS)
LOG_DIR = RESULTS_DIR / "logs"; LOG_DIR.mkdir(parents=True, exist_ok=True)
KEEP_GOING = MODE != "full"
TIMINGS = []
# start from clean output folders, so that no file from an earlier run or mode is verified by mistake
if True:
    for d in (RESULTS_DIR, FIG_DIR):
        for p in d.glob("*"):
            if p.is_file():
                p.unlink()
        for p in (FIG_DIR / "manuscript_final").glob("*"):
            p.unlink()
    for p in LOG_DIR.glob("*"):
        p.unlink()

def _short(line, n=150):
    line = line.rstrip()
    return line if len(line) <= n else line[:n - 3] + "..."

def run_steps(ids):
    # run pipeline steps by id; concise live log; returns True if all succeeded
    env = dict(os.environ, **PIPE_ENV)
    all_ok = True
    for sid in ids:
        _, kind, script, desc = STEPS[sid]
        cmd = ["Rscript", str(REPO_DIR / script)] if kind == "R" else [sys.executable, str(REPO_DIR / script)]
        log = LOG_DIR / f"{STEP_IDS.index(sid) + 1:02d}_{sid}.log"
        print(f"[{STEP_IDS.index(sid) + 1:2d}/{len(STEP_IDS)}] {sid:7s} {desc}", flush=True)
        t = time.time(); last_beat = t
        with open(log, "w") as lf:
            proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), env=env, stdout=lf, stderr=subprocess.STDOUT)
            while proc.poll() is None:
                time.sleep(1)
                if time.time() - last_beat >= HEARTBEAT_SECONDS:
                    last_beat = time.time()
                    print(f"          ... still running ({(last_beat - t) / 60:.1f} min)", flush=True)
        dt = time.time() - t
        ok = proc.returncode == 0
        text = log.read_text(errors="replace").splitlines()
        checks = [l for l in text if re.search(r"^\s*(\[non-strict mode\]|REPRO|Reproduction|ANCHOR\b)|->\s*(OK|PASS|MISMATCH|FAIL)", l)][:3]
        for l in checks:
            print("          " + _short(l.strip()))
        print(f"          -> {'ok' if ok else 'FAILED (exit %d)' % proc.returncode}  {dt / 60:.2f} min  [log: {log.name}]", flush=True)
        TIMINGS.append({"step": sid, "script": script, "status": "ok" if ok else f"failed ({proc.returncode})", "minutes": round(dt / 60, 2)})
        if not ok:
            all_ok = False
            print("          last log lines:\n          " + "\n          ".join(_short(l) for l in text[-12:]))
            if not KEEP_GOING:
                raise RuntimeError(f"Step {sid} failed in '{MODE}' mode; see {log}. Fix the cause and re-run this cell "
                                   "(earlier steps need not be repeated).")
    return all_ok

print("Steps:", " ".join(STEP_IDS))

### 4.1 Cohort construction (patient-level files in `work/`, never displayed) and flow counts

In [ ]:
_ok = run_steps(["COHORTS", "FLOW"])

### 4.2 Main landmark analysis (Table 2), window sensitivity and model comparison, equivalence framing (A01–A03)

In [ ]:
_ok = run_steps(["A01", "A02", "A03"])

### 4.3 Case definitions and subtypes, heterogeneity and standardization (A04, A05R, A05)

In [ ]:
_ok = run_steps(["A04", "A05R", "A05"])

### 4.4 Piecewise-exponential suite, split-sample validation, extended standardization, smoothed hazards (A06–A09)

In [ ]:
_ok = run_steps(["A06", "A07", "A08", "A09", "A09MD"])

### 4.5 Data quality, cross-tumour comparison (Table 3) and lung LCNEC rate ratios (A10–A12)

In [ ]:
_ok = run_steps(["A10", "A11", "A12"])

### 4.6 Manuscript figures (Figures 1–3, Supplementary Figures S1–S7)

In [ ]:
_ok = run_steps(["FIG"])
import pandas as pd
timing = pd.DataFrame(TIMINGS)
print(f"Pipeline: {sum(t['status'] == 'ok' for t in TIMINGS)}/{len(TIMINGS)} steps ok, {timing['minutes'].sum():.1f} min in total")
timing

## 5. Verification

* `tests/check_anchors.py` — the key numbers printed in the manuscript (Table 2 landmark probabilities, numbers at risk, cohort sizes, plateau, rate ratios, comparators, lung LCNEC). The table below lists **manuscript value vs reproduced value vs status** for every anchor.
* `tests/compare_with_reference.py` — every produced result JSON (and `figure_data.json`) against the authors' aggregate `reference_results/` (tolerance 1e-6; seeds exact; the fields suppressed there as small cells are skipped).

Expected outcome: **`full`** — all anchors PASS and 17/17 reference files reproduced. **`quick`** — point-estimate anchors pass, but bootstrap-based quantities (e.g. the plateau CI) and the reference comparison differ because B = 40 instead of 2,000. **`synthetic`** — the data are random, so anchor failures and reference mismatches are **expected** and do not indicate an error.

In [ ]:
soft = MODE != "full"
r = subprocess.run([sys.executable, str(REPO_DIR / "tests" / "check_anchors.py"), "--soft"], cwd=str(REPO_DIR),
                   env=dict(os.environ, **PIPE_ENV), capture_output=True, text=True)
anchor_summary = [l for l in r.stdout.splitlines() if "anchors met" in l]
print("tests/check_anchors.py:", anchor_summary[0] if anchor_summary else r.stdout[-500:] + r.stderr[-500:])

# the same anchors, as a table (imports the test module with this run's configuration)
for m in ("config", "nec_check_anchors"):
    sys.modules.pop(m, None)
sys.path.insert(0, str(REPO_DIR / "python"))
_spec = importlib.util.spec_from_file_location("nec_check_anchors", REPO_DIR / "tests" / "check_anchors.py")
ca = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(ca)

def _fmt(x):
    if isinstance(x, float):
        return f"{x:.4g}" if abs(x) < 1e4 else f"{x:,.0f}"
    return str(x)

rows = []
for group, label, getter, expected, tol in ca.A:
    try:
        got = getter(); ok = ca.close(got, expected, tol)
    except Exception as e:
        got, ok = f"<{type(e).__name__}>", False
    rows.append({"group": group, "quantity": label, "manuscript value": _fmt(expected), "reproduced value": _fmt(got),
                 "status": "PASS" if ok else ("expected mismatch (synthetic)" if MODE == "synthetic" else "FAIL")})
anchors = pd.DataFrame(rows)
N_ANCHORS_PASS = int((anchors["status"] == "PASS").sum())
ANCHORS_ALL_PASS = N_ANCHORS_PASS == len(anchors)
print(f"{N_ANCHORS_PASS}/{len(anchors)} anchors met" + ("  -> ALL ANCHORS PASSED" if ANCHORS_ALL_PASS else ""))
if MODE == "synthetic":
    print("Synthetic data: the manuscript values cannot be reproduced from random data; mismatches are expected.")
elif MODE == "quick":
    print("Quick mode: bootstrap-dependent anchors (plateau CI) may differ because B = 40.")
pd.set_option("display.max_rows", 200)
anchors.style.apply(lambda s: ["color: #1b7a1b" if v == "PASS" else ("color: #b26b00" if "expected" in v else "color: #c00000")
                               for v in s], subset=["status"]).hide(axis="index")

In [ ]:
report = LOG_DIR / "compare_with_reference.json"
r = subprocess.run([sys.executable, str(REPO_DIR / "tests" / "compare_with_reference.py"), "--soft", "--report", str(report)],
                   cwd=str(REPO_DIR), env=dict(os.environ, **PIPE_ENV), capture_output=True, text=True)
summary_line = [l for l in r.stdout.splitlines() if "reference files reproduced" in l]
print("tests/compare_with_reference.py:", summary_line[0] if summary_line else r.stdout[-500:] + r.stderr[-500:])
rep = json.loads(report.read_text())
cmp_rows = []
for name, d in rep["files"].items():
    cmp_rows.append({"reference file": name + ".json", "values compared": d.get("values_compared"),
                     "max |difference|": d.get("max_abs_diff"), "mismatches": len(d.get("mismatches", [])) if d["status"] != "missing" else None,
                     "status": d["status"].upper()})
compare = pd.DataFrame(cmp_rows)
REFERENCE_ALL_PASS = bool(rep.get("all_pass"))
print(f"{(compare['status'] == 'PASS').sum()}/{len(compare)} reference files reproduced"
      + ("  -> IDENTICAL TO THE AUTHORS' VERIFIED RUN" if REFERENCE_ALL_PASS else ""))
if MODE != "full":
    print(f"'{MODE}' mode: differences from the reference are expected "
          + ("(random synthetic data)." if MODE == "synthetic" else "(bootstrap size B = %s instead of 2,000)." % os.environ.get("NEC_BOOT", "40")))
compare

## 6. Figures and outputs

Figures 1–3 of the manuscript (300-dpi PNG; vector PDFs are written next to them). In `synthetic` mode they carry a "SYNTHETIC DATA" watermark. Without the Arial font (e.g. in Colab) the figures fall back to DejaVu Sans; this does not change any number.

In [ ]:
from IPython.display import Image, display, Markdown
MS_DIR = FIG_DIR / "manuscript_final"
for i in (1, 2, 3):
    p = MS_DIR / f"Figure{i}.png"
    if p.exists():
        display(Markdown(f"**Figure {i}** — `{p.relative_to(REPO_DIR)}`"))
        display(Image(filename=str(p), width=900))
    else:
        print(f"Figure {i} not found ({p})")

In [ ]:
def _list(d, pattern="*"):
    files = sorted(p for p in Path(d).glob(pattern) if p.is_file())
    for p in files:
        print(f"  {str(p.relative_to(REPO_DIR)):58s} {p.stat().st_size / 1024:9.1f} KB")
    return files

print("Aggregate results (results/):"); res_files = _list(RESULTS_DIR)
print("Step logs and check reports (results/logs/):"); log_files = _list(LOG_DIR)
print("Manuscript figures (figures/output/manuscript_final/):"); fig_files = _list(MS_DIR)
other_figs = sorted(p for p in FIG_DIR.glob("*") if p.is_file())
print(f"Analysis figures (figures/output/): {len(other_figs)} files")
print("See docs/output_map.md for the mapping of each file to Tables 2-3, Figures 1-3, Supplementary Tables S2-S43 and Figures S1-S7.")

### Optional: download aggregate outputs

Set `MAKE_ZIP` (Colab form) or `NEC_MAKE_ZIP=1` (local) to pack the **aggregate** outputs — `results/*.json|*.md`, the aggregate hazard grid `A09_hazard_grid.csv`, the check reports and the figures — into a zip. Derived cohorts (`work/`), exports and step logs are never included; the cell refuses any file from the data or work folders.

In [ ]:
ZIP_PATH = None
if MAKE_ZIP:
    ZIP_PATH = RUNTIME_DIR / f"colorectal_nec_aggregate_outputs_{MODE}.zip"
    allowed_ext = {".json", ".md", ".png", ".pdf", ".csv"}
    forbidden = [str(WORK_DIR.resolve()), str(Path(DATA_DIR_EFFECTIVE).resolve())]
    candidates = ([p for p in RESULTS_DIR.glob("*") if p.is_file()] + [LOG_DIR / "compare_with_reference.json"] +
                  [p for p in FIG_DIR.rglob("*") if p.is_file()])
    n = 0
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
        for p in candidates:
            rp = str(p.resolve())
            if (not p.exists() or p.suffix.lower() not in allowed_ext or any(rp.startswith(f) for f in forbidden)
                    or p.name.startswith("cohort_") or p.name == "flow_all.csv"):
                continue
            if p.suffix.lower() == ".csv" and p.name != "A09_hazard_grid.csv":
                continue          # only the known aggregate CSV
            z.write(p, arcname=str(p.relative_to(REPO_DIR))); n += 1
        if MODE == "synthetic":
            z.writestr("SYNTHETIC_RESULTS_NOT_THE_PAPER.txt", "These outputs were computed from random synthetic data.\n")
    print(f"{n} aggregate files -> {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(ZIP_PATH))
else:
    print("MAKE_ZIP is off: no archive created.")

### Cleanup (run this cell when you are done)

Deletes the derived patient-level cohorts (`work/`), the runtime copy of the exports (Colab) and any synthetic exports. Aggregate results and figures are kept. In Colab, finally use *Runtime ▸ Disconnect and delete runtime*; your original exports on Drive are not touched.

In [ ]:
removed = []
if CLEANUP_AT_END:
    targets = [WORK_DIR]
    if COPIED_DATA_DIR is not None:
        targets.append(COPIED_DATA_DIR)
    if MODE == "synthetic":
        targets.append(DATA_DIR_EFFECTIVE)
    for t in targets:
        t = Path(t)
        if t.exists():
            shutil.rmtree(t); removed.append(str(t))
    for p in REPO_DIR.rglob("__pycache__"):
        shutil.rmtree(p, ignore_errors=True)
    print("Deleted:\n  " + "\n  ".join(removed) if removed else "Nothing to delete.")
    print("work/ present after cleanup:", WORK_DIR.exists())
else:
    print(f"CLEANUP_AT_END is off. Patient-level cohorts remain in {WORK_DIR}; delete them when finished.")
if IN_COLAB:
    print("Colab: end the session with Runtime > Disconnect and delete runtime.")
print(f"\nSummary: mode={MODE}  anchors {N_ANCHORS_PASS}/{len(anchors)}  reference files reproduced: {REFERENCE_ALL_PASS}")